# Exp10.2 — L1 Membrane Memory and Subthreshold-Evidence Recovery

Aggregation-only notebook for `d1_bb_l1_mem_shift_sweep_v1`. Stage A changes only L1 membrane dynamics with frozen learned weights; Stage B retrains the same D1+BB A2 for each L1 membrane shift.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_10_2_l1_membrane_memory' / 'd1_bb_l1_mem_shift_sweep_v1'
audit = json.loads((root / 'audit.json').read_text())
audit


## Stage A — frozen-weight membrane replay

Primary question: does increasing L1 membrane memory make `L1 spike Fixed250 - L1 pre-reset Fixed250` less negative while preserving the pre-reset probe?

In [ ]:
stage_a = pd.read_csv(root / 'stage_a_replay_summary.csv')
display(stage_a)

stage_a_contrast = pd.read_csv(root / 'stage_a_replay_shift_contrast_summary.csv')
display(stage_a_contrast)

sanity = pd.read_csv(root / 'stage_a_shift1_replay_sanity.csv')
display(sanity)


## Stage B — end-to-end retraining

The table below keeps Stage A and Stage B side-by-side. Seeds are optimization replicates on one locked user split.

In [ ]:
runs = pd.read_csv(root / 'all_runs.csv')
summary = pd.read_csv(root / 'all_summary.csv')
display(summary)

contrast = pd.read_csv(root / 'all_shift_contrast_summary.csv')
display(contrast)


## Information-path view

In [ ]:
cols = [
    'stage','l1_mem_shift','seed','native_test_ba',
    'l1_pre_reset_fixed250_ba','l1_spike_fixed250_ba','l1_quantization_delta',
    'l2_pre_reset_fixed250_ba','l2_spike_fixed250_ba','l2_spike_whole_ba',
    'l2_temporal_ordering_gain'
]
display(runs[cols].sort_values(['stage','l1_mem_shift','seed']))


## Firing/dead/saturation diagnostics

Longer membrane memory is useful only if spike decodability improves without driving the population into a high-rate or saturated regime.

In [ ]:
activity = pd.read_csv(root / 'all_activity_summary.csv')
display(activity)


## Frozen replay vs retrained dynamics

In [ ]:
comparison = pd.read_csv(root / 'e2e_minus_replay.csv')
display(comparison.sort_values(['l1_mem_shift','seed']))


## Interpretation guide

- **Recovery:** L1 spike Fixed250 rises while L1 pre-reset Fixed250 is preserved, so the quantization delta becomes less negative.
- **Fake gap closure:** the gap shrinks only because pre-reset decodability falls.
- **Temporal smearing:** L1 gap may improve but L2 Fixed250/native BA falls as membrane memory becomes too long.
- **Persistent/saturated regime:** high-rate or >=95%-active neuron fractions rise strongly with shift.
- **Training adaptation:** E2E outperforms frozen replay at the same shift, showing weights can reorganize around the longer membrane dynamics.